# Flashmind PDM · GIFT-Eval

Evaluate [Femto](https://www.flashmind.com/models/pdm#pdm-femto),
[Pico](https://www.flashmind.com/models/pdm#pdm-pico), or
[Nano](https://www.flashmind.com/models/pdm#pdm-nano) from
[Flashmind UAE](https://www.flashmind.com/) through the
[public API](https://www.flashmind.com/docs/api), using the
[flashpdm Python client](https://www.flashmind.com/docs/python).
Use Python 3.12 and an evaluator
API key with sufficient limits.

The notebook loads the official GIFT datasets and passes each evaluation window
to `flashpdm`, which handles batching, padding, concurrent API requests and
forecast recovery. Forecasts are saved locally, then scored with GIFT/GluonTS
and exported as `all_results.csv` and `config.json` for submission.


In [ ]:
%pip install -q "flashpdm[batch]==0.1.11" "pyarrow==17.0.0" "salesforce-gift-eval @ git+https://github.com/SalesforceAIResearch/gift-eval.git@cfaefe09c6bb4195b38c720bc3c8ea382048241d"


In [ ]:
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from getpass import getpass
from itertools import tee
from pathlib import Path
from threading import Lock

import httpx
import numpy as np
import pandas as pd
from flashpdm import Client
from flashpdm.batching import ForecastPool
from gift_eval.data import Dataset
from gluonts.ev.metrics import MAE, MAPE, MASE, MSE, MSIS, ND, NRMSE, RMSE, SMAPE, MeanWeightedSumQuantileLoss
from gluonts.model import evaluate_model
from gluonts.model.forecast import QuantileForecast
from gluonts.time_feature import get_seasonality
from huggingface_hub import snapshot_download
from joblib import Parallel, delayed, parallel_config

MODEL = os.environ.get("FLASHPDM_MODEL", "pdm-nano")  # pdm-femto, pdm-pico, pdm-nano
REVISIONS = {
    'pdm-femto': '46bb0bde1c836a6b1a82e6d0108f766adb47c8639cf9c695637c583a617fe06c.2026-10-27',
    'pdm-pico': 'cf380714c0e6eda4a43a1e62a12cf228b3c0795fa21be7078ddaf6b866c501bd.2026-10-27',
    'pdm-nano': '48cc3fd4918228d313179822d616fca5c282c5bcc9e59d55bc33c31b2c98eb5e.2026-10-27',
}
API_KEY = os.environ.get("FLASHMIND_API_KEY") or getpass("Flashmind API key: ")
CONCURRENCY = 128 if MODEL == "pdm-nano" else 32
SCORING_WORKERS = min(16 if MODEL == "pdm-nano" else 8, os.cpu_count() or 1)
OUTPUT = Path("outputs") / MODEL
CACHE = Path(os.environ.get("FLASHPDM_BATCH_DIR", OUTPUT / "batches"))
QUANTILES = [i / 10 for i in range(1, 10)]


## Official configurations

Data uses the Hugging Face cache, or an existing `GIFT_EVAL` directory.
The official baseline defines the 97 configurations and normalization.


In [ ]:
GIFT_SOURCE = "https://raw.githubusercontent.com/SalesforceAIResearch/gift-eval/cfaefe09c6bb4195b38c720bc3c8ea382048241d"
os.environ.setdefault("GIFT_EVAL", snapshot_download("Salesforce/GiftEval", repo_type="dataset") if not os.environ.get("GIFT_EVAL") else os.environ["GIFT_EVAL"])
OUTPUT.mkdir(parents=True, exist_ok=True)
with httpx.Client(timeout=60) as files:
    response = files.get(f"{GIFT_SOURCE}/notebooks/dataset_properties.json")
    response.raise_for_status()
    properties = response.json()
    response = files.get(f"{GIFT_SOURCE}/results/Seasonal_Naive/all_results.csv")
    response.raise_for_status()
    (OUTPUT / "seasonal_naive.csv").write_bytes(response.content)
baseline = pd.read_csv(OUTPUT / "seasonal_naive.csv").set_index("dataset")
aliases = {"saugeen":"saugeenday", "temperature_rain":"temperature_rain_with_missing", "kdd_cup_2018":"kdd_cup_2018_with_missing", "car_parts":"car_parts_with_missing"}
directories = {p.name.lower():p for p in Path(os.environ["GIFT_EVAL"]).iterdir() if p.is_dir()}
configs = []
for key in baseline.index:
    name, frequency, term = key.split("/")
    directory = directories[aliases.get(name, name)]
    dataset = f"{directory.name}/{frequency}" if (directory / frequency).is_dir() else directory.name
    configs.append((dataset, term, key))
assert len(configs) == len(set(baseline.index)) == 97


## GIFT adapter

Targets remain joint; past covariates stay hidden from scoring. Point metrics use
the median. Nonnegative/count projection uses observed history only.


In [ ]:
class GiftPredictor:
    def __init__(self, forecasts, dataset):
        self.forecasts = forecasts
        self.prediction_length = dataset.prediction_length
        self.use_covariates = bool(dataset.past_feat_dynamic_real_dim)

    def predict(self, inputs):
        entries, targets, covariates = tee(inputs, 3)
        contexts = (np.atleast_2d(np.asarray(item["target"], dtype=np.float32)) for item in targets)
        past = (np.asarray(item["past_feat_dynamic_real"], dtype=np.float32)
                if self.use_covariates and "past_feat_dynamic_real" in item else None for item in covariates)
        forecasts = self.forecasts.forecast_quantiles(contexts, self.prediction_length,
            past_covariates=past, quantile_levels=QUANTILES)
        for item, bands in zip(entries, forecasts):
            history = np.atleast_2d(np.asarray(item["target"], dtype=np.float32))
            bands = bands.astype(np.float32)
            for channel, values in enumerate(history):
                observed = values[np.isfinite(values)]
                if observed.size and np.all(observed >= 0):
                    bands[channel] = np.maximum(bands[channel], 0)
                    if observed.size >= 8 and np.all(observed == np.rint(observed)):
                        bands[channel] = np.rint(bands[channel])
            values = np.concatenate([bands, bands[..., 4:5]], axis=-1).transpose(2, 1, 0)
            if np.asarray(item["target"]).ndim == 1:
                values = values[..., 0]
            yield QuantileForecast(values, start_date=item["start"] + history.shape[-1],
                forecast_keys=[str(q) for q in QUANTILES] + ["mean"], item_id=item.get("item_id"))

METRICS = [MSE(forecast_type="mean"), MSE(forecast_type=.5), MAE(), MASE(), MAPE(),
           SMAPE(), MSIS(), RMSE(), NRMSE(), ND(), MeanWeightedSumQuantileLoss(quantile_levels=QUANTILES)]


## Forecast, then score

Forecasts are saved before CPU scoring. Rerunning reuses validated batches.
Three whole-batch attempts are enabled here; uncertain failures may repeat server work.


In [ ]:
dataset_lock = Lock()

def forecast_config(item, forecasts):
    name, term, key = item
    with dataset_lock:
        dataset = Dataset(name=name, term=term, to_univariate=False)
    for _ in GiftPredictor(forecasts, dataset).predict(dataset.test_data.input):
        pass
    return key

def score_config(item, cached):
    name, term, key = item
    try:
        dataset = Dataset(name=name, term=term, to_univariate=False)
        with cached:
            result = evaluate_model(GiftPredictor(cached, dataset), test_data=dataset.test_data,
                metrics=METRICS, batch_size=512, axis=None, mask_invalid_label=True,
                allow_nan_forecast=False, seasonality=get_seasonality(dataset.freq))
        values = {column:float(result[column].iloc[0]) for column in result.columns}
        assert all(np.isfinite(value) for value in values.values()), key
        return key, values, None
    except Exception as error:
        return key, None, str(error)

ready, failures, measured = [], {}, {}
with Client(api_key=API_KEY, max_retries=0) as client, ThreadPoolExecutor(SCORING_WORKERS) as jobs:
    with ForecastPool(client, model=MODEL, revision=REVISIONS[MODEL], concurrency=CONCURRENCY, cache_dir=CACHE, max_attempts=3) as forecasts:
        print(f"Context: {forecasts.capabilities['max_context']}; revision: {forecasts.revision}")
        (OUTPUT / "revision.txt").write_text(forecasts.revision)
        pending = {jobs.submit(forecast_config, item, forecasts):item for item in configs}
        for future in as_completed(pending):
            item = pending[future]
            try:
                future.result()
                ready.append(item)
            except Exception as error:
                failures[item[2]] = str(error)
            print(f"Forecasts saved: {len(ready)}/{len(configs)}; failures: {len(failures)}")
        cached = forecasts.cached()
        (OUTPUT / "batches-summary.json").write_text(json.dumps(forecasts.stats, indent=2))

with parallel_config(backend="loky", inner_max_num_threads=1):
    scores = Parallel(n_jobs=SCORING_WORKERS, return_as="generator_unordered")(
        delayed(score_config)(item, cached) for item in ready)
    for key, values, error in scores:
        if error:
            failures[key] = error
        else:
            measured[key] = values
            (OUTPUT / (key.replace("/", "__") + ".json")).write_text(json.dumps(values))
        print(f"Scored: {len(measured)}/{len(configs)}; failures: {len(failures)}")
(OUTPUT / "failures.json").write_text(json.dumps(failures, indent=2))
assert not failures and set(measured) == set(baseline.index), failures


## Scores and submission


In [ ]:
assert len(measured) == 97 and set(measured) == set(baseline.index)
MODEL_NAME = "PDM-" + MODEL.removeprefix("pdm-").title()
rows = [dict(dataset=key, model=MODEL_NAME,
             **{f"eval_metrics/{column}":value for column, value in values.items()},
             domain=properties[key.split("/")[0]]["domain"],
             num_variates=properties[key.split("/")[0]]["num_variates"])
        for key, values in measured.items()]
results = pd.DataFrame(rows).sort_values("dataset")
for name, column in [("MASE", "eval_metrics/MASE[0.5]"), ("CRPS", "eval_metrics/mean_weighted_sum_quantile_loss")]:
    ratios = results.set_index("dataset")[column] / baseline[column]
    assert np.isfinite(ratios).all() and (ratios > 0).all()
    print(f"Normalized {name}: {np.exp(np.log(ratios).mean()):.6f}")
submission = OUTPUT / "results" / MODEL_NAME
submission.mkdir(parents=True, exist_ok=True)
results.to_csv(submission / "all_results.csv", index=False)
(submission / "config.json").write_text(json.dumps({
    "model":MODEL_NAME, "model_type":"zero-shot", "model_link":f"https://www.flashmind.com/models/pdm#{MODEL}",
    "org":"Flashmind UAE", "testdata_leakage":"No", "replication_code_available":"Yes",
}, indent=2) + "\n")
submission
